# Homework 1 — NOAA Storm Events (`rel-noaa-storms`) & `zone-damage` Baselines

**Dataset:** `rel-noaa-storms` (NOAA Storm Events, 2006–2025, 8 relational tables, 3.72M tuples, non-star multi-cycle schema)

**Task (`zone-damage`):** *At quarterly anchor timestamp $t$, predict whether NWS county/forecast zone `zone_id` will experience a damaging severe weather event (`total_damage_usd > 0`) over the next 90 days $[t, t + 90\text{d})$.*

Outline
1. Load the RelBench database (`hw-1/output`) and inspect tables, keys, types, and time columns
2. Load the `zone-damage` binary classification task and inspect temporal train / val / test splits & class balance
3. Baseline 1: Global & per-zone historical majority / rate
4. Baseline 2: XGBoost on the target table only (`zone_id` + timestamp)
5. Baseline 3: Deep Feature Synthesis (automatic multi-hop schema-driven DFS + manual multi-table join-aggregate features, strictly leak-free)
6. Compare ROC-AUC across all baselines

## 1. The Relational Database

In [1]:
import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import relbench

dataset_dir = Path("output")
dataset = relbench.load_dataset(dataset_dir)
db = dataset.get_db()  # rows at/after test_timestamp (2024-01-01) are hidden

print("Tasks:          ", dataset.get_task_names())
print("val timestamp:  ", dataset.val_timestamp)
print("test timestamp: ", dataset.test_timestamp)
print(db)

Tasks:           ['zone-damage']
val timestamp:   2022-01-01 00:00:00
test timestamp:  2024-01-01 00:00:00
Database(states=69, zones=8618, places=92723, episodes=192308, episode_zones=745767, events=1139980, event_locations=1132541, fatalities=15433)


In [2]:
def print_schema(db, title="RELATIONAL DB SCHEMA (rel-noaa-storms)"):
    print(title)
    for i, (name, table) in enumerate(db.table_dict.items(), start=1):
        header_parts = [
            f"{i}. {name} ({len(table.df):,})",
            f"PK: {table.pkey_col}",
        ]
        if table.fkey_col_to_pkey_table:
            fk_str = ", ".join(
                f"{col} -> {target}"
                for col, target in table.fkey_col_to_pkey_table.items()
            )
            header_parts.append(f"FK: {fk_str}")
        if table.time_col is not None:
            header_parts.append(f"TIME: {table.time_col}")
        cols_str = ", ".join(f"{col} ({dtype})" for col, dtype in table.df.dtypes.items())
        print(" | ".join(header_parts))
        print(f"   cols: {cols_str}")

print_schema(db)

RELATIONAL DB SCHEMA (rel-noaa-storms)
1. states (69) | PK: state_id
   cols: state_id (int64), state (object), state_fips (int64)
2. zones (8,618) | PK: zone_id | FK: state_id -> states
   cols: zone_id (int64), state_id (int64), cz_type (object), cz_name (object), cz_fips (int64)
3. places (92,723) | PK: place_id | FK: zone_id -> zones
   cols: place_id (int64), zone_id (int64), place_name (object)
4. episodes (192,308) | PK: episode_id | FK: state_id -> states | TIME: begin_date_time
   cols: episode_id (int64), state_id (int64), begin_date_time (datetime64[us]), episode_narrative (object)
5. episode_zones (745,767) | PK: episode_zone_id | FK: episode_id -> episodes, zone_id -> zones | TIME: begin_date_time
   cols: episode_zone_id (int64), episode_id (int64), zone_id (int64), begin_date_time (datetime64[us])
6. events (1,139,980) | PK: event_id | FK: episode_zone_id -> episode_zones | TIME: begin_date_time
   cols: event_id (int64), episode_zone_id (int64), begin_date_time (datetim

In [3]:
states_df = db.table_dict["states"].df
zones_df = db.table_dict["zones"].df
places_df = db.table_dict["places"].df
episodes_df = db.table_dict["episodes"].df
episode_zones_df = db.table_dict["episode_zones"].df
events_df = db.table_dict["events"].df
event_locations_df = db.table_dict["event_locations"].df
fatalities_df = db.table_dict["fatalities"].df

events_df.head()

,event_id,episode_zone_id,begin_date_time,event_type,source,magnitude,magnitude_type,damage_property_usd,damage_crops_usd,total_damage_usd,injuries_direct,injuries_indirect,deaths_direct,deaths_indirect,event_narrative
0,0,3,2006-01-01,Winter Weather,"AWOS,ASOS,MESONET,ETC",NaN,NONE,0.0,0.0,0.0,0,0,0,0,
1,1,4,2006-01-01,Winter Storm,"AWOS,ASOS,MESONET,ETC",NaN,NONE,0.0,0.0,0.0,0,0,0,0,
2,2,5,2006-01-01,Winter Storm,"AWOS,ASOS,MESONET,ETC",NaN,NONE,0.0,0.0,0.0,0,0,0,0,
3,3,6,2006-01-01,Winter Storm,"AWOS,ASOS,MESONET,ETC",NaN,NONE,0.0,0.0,0.0,0,0,0,0,
4,4,7,2006-01-01,Winter Storm,"AWOS,ASOS,MESONET,ETC",NaN,NONE,0.0,0.0,0.0,0,0,0,0,


## 2.1. The Classification Prediction Task (`zone-damage`)
Entity-level binary classification on the zones table (7,703 active NWS county/forecast zones with ≥ 5 events in the training window). At each non-overlapping quarterly anchor timestamp t (Jan 1, Apr 1, Jul 1, Oct 1), predict whether zone_id will experience at least one damaging severe weather event (total_damage_usd > 0, CPI-adjusted) within the next 90 days [t, t + 90d), using only historical database records strictly before t (time_col < t).

In [4]:
task = dataset.load_task("zone-damage")
print(
    "type:", task.task_type,
    "| entity:", task.entity_col,
    "| target:", task.target_col,
    "| horizon:", task.timedelta,
)

START_YEAR = 2006  # Consistent episode_id, narratives, and all 48 NOAA event types
END_YEAR = 2025    # Last complete calendar year (avoids partial 2026 backfill lag)
VAL_TIMESTAMP = "2022-01-01"
TEST_TIMESTAMP = "2024-01-01"
HORIZON_DAYS = 90  # Quarterly non-overlapping prediction horizon
DAMAGE_THRESHOLD_USD = 0.0

quarters = (END_YEAR - START_YEAR) * 4

import datetime
start_date = datetime.date(START_YEAR, 1, 1)
end_date = datetime.date(END_YEAR, 1, 1)

for quarter in range(quarters):

    year = START_YEAR + int(quarter / 4)
    month = 1 + (quarter % 4) * 3

    quart_start_date = datetime.date(year, month, 1)
    horz_end_date = quart_start_date + datetime.timedelta(days=HORIZON_DAYS)

    # print(quart_start_date, horz_end_date)

train_df = task.get_table("train").df

global_mean = train_df["has_damage"].mean()
zone_mean = train_df.groupby("zone_id")["has_damage"].mean()

type: TaskType.BINARY_CLASSIFICATION | entity: zone_id | target: has_damage | horizon: 90 days 00:00:00


In [5]:
task = dataset.load_task("zone-damage")
print(
    "type:", task.task_type,
    "| entity:", task.entity_col,
    "| target:", task.target_col,
    "| horizon:", task.timedelta,
)

train_table = task.get_table("train")
val_table = task.get_table("val")
test_table = task.get_table("test", mask_input_cols=False)

train_df = train_table.df
val_df = val_table.df
test_df = test_table.df

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    pos_rate = df["has_damage"].mean() * 100
    print(
        f"{name:5s} {str(df.shape):12s} | "
        f"{df['timestamp'].min().date()} -> {df['timestamp'].max().date()} | "
        f"pos rate: {pos_rate:5.2f}%"
    )

train_df.head()

type: TaskType.BINARY_CLASSIFICATION | entity: zone_id | target: has_damage | horizon: 90 days 00:00:00
train (462180, 3)  | 2007-01-01 -> 2021-10-01 | pos rate: 18.45%
val   (61624, 3)   | 2022-01-01 -> 2023-10-01 | pos rate: 14.11%
test  (61624, 3)   | 2024-01-01 -> 2025-10-01 | pos rate: 14.05%


,timestamp,zone_id,has_damage
0,2007-01-01,0,1
1,2007-01-01,1,0
2,2007-01-01,2,1
3,2007-01-01,3,1
4,2007-01-01,4,0


In [6]:
zone_id = 1077
zones_df[zones_df["zone_id"] == zone_id].head()
epi_by_zone = episode_zones_df[episode_zones_df["zone_id"] == zone_id]
events_df[events_df["episode_zone_id"].isin(epi_by_zone["episode_zone_id"])]

,event_id,episode_zone_id,begin_date_time,event_type,source,magnitude,magnitude_type,damage_property_usd,damage_crops_usd,total_damage_usd,injuries_direct,injuries_indirect,deaths_direct,deaths_indirect,event_narrative
539,539,522,2006-01-02 09:00:00,High Wind,TRAINED SPOTTER,50.0,MG,41069.001559,0.0,41069.001559,0,0,0,0,
877,877,804,2006-01-05 10:00:00,Strong Wind,"AWOS,ASOS,MESONET,ETC",50.0,MG,16427.600624,0.0,16427.600624,0,0,0,0,
2519,2519,2323,2006-01-23 09:50:00,High Wind,"AWOS,ASOS,MESONET,ETC",50.0,MG,24641.400935,0.0,24641.400935,0,0,0,0,
37536,37536,27822,2006-07-21 00:00:00,Heat,"AWOS,ASOS,MESONET,ETC",NaN,NONE,0.000000,0.0,0.000000,6,0,3,0,
55852,55852,42480,2006-12-27 10:00:00,Strong Wind,Mesonet,39.0,MG,1642.760062,0.0,1642.760062,0,0,0,0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1125120,1125120,735045,2023-08-20 05:00:00,Tropical Storm,Mesonet,NaN,NONE,0.000000,0.0,0.000000,0,0,0,0,Tropical Storm Hilary produced widespread trop...
1127327,1127327,736501,2023-08-28 11:00:00,Excessive Heat,ASOS,NaN,NONE,0.000000,0.0,0.000000,0,0,0,0,Very hot daytime temperatures occurred across ...
1134238,1134238,740944,2023-10-18 12:00:00,Heat,Mesonet,NaN,NONE,0.000000,0.0,0.000000,0,0,0,0,High temperatures across portions of the San D...
1134805,1134805,741347,2023-10-29 13:00:00,High Wind,Utility Company,56.0,MG,0.000000,0.0,0.000000,0,0,0,0,Gusty Santa Ana winds occurred across the San ...


In [7]:
from sklearn.metrics import roc_auc_score

y_train = train_df["has_damage"].to_numpy(dtype=int)
y_val = val_df["has_damage"].to_numpy(dtype=int)
y_test = test_df["has_damage"].to_numpy(dtype=int)

results_table = {}

def report(name, val_pred, test_pred):
    val_pred = np.asarray(val_pred, dtype=float)
    test_pred = np.asarray(test_pred, dtype=float)
    val_auc = task.evaluate(val_pred, val_table)["roc_auc"]
    test_auc = roc_auc_score(y_test, test_pred)
    results_table[name] = {"Val ROC-AUC": val_auc, "Test ROC-AUC": test_auc}
    print(f"{name:40s} | Val AUC: {val_auc:.4f} | Test AUC: {test_auc:.4f}")

## 3. Baseline 1 — Historical Majority & Historical Positive Rate

In [8]:
global_rate = float(y_train.mean())
global_maj = int(global_rate >= 0.5)

zone_rates_tr = train_df.groupby("zone_id")["has_damage"].mean()
zone_maj_tr = (zone_rates_tr >= 0.5).astype(int)

train_val_df = pd.concat([train_df, val_df], ignore_index=True)
zone_rates_trval = train_val_df.groupby("zone_id")["has_damage"].mean()
zone_maj_trval = (zone_rates_trval >= 0.5).astype(int)

report(
    "1a. Global Historical Majority",
    np.full(len(val_df), global_maj),
    np.full(len(test_df), global_maj)
)

report(
    "1b. Per-Zone Historical Majority",
    val_df["zone_id"].map(zone_maj_tr).fillna(global_maj).to_numpy(),
    test_df["zone_id"].map(zone_maj_trval).fillna(global_maj).to_numpy(),
)
report(
    "1c. Per-Zone Historical Rate",
    val_df["zone_id"].map(zone_rates_tr).fillna(global_rate).to_numpy(),
    test_df["zone_id"].map(zone_rates_trval).fillna(global_rate).to_numpy(),
)

1a. Global Historical Majority           | Val AUC: 0.5000 | Test AUC: 0.5000
1b. Per-Zone Historical Majority         | Val AUC: 0.6375 | Test AUC: 0.6318
1c. Per-Zone Historical Rate             | Val AUC: 0.8068 | Test AUC: 0.8037


## 4. Baseline 2 — XGBoost on Target Table Only

In [9]:
import xgboost as xgb

def target_table_features(df):
    ts = pd.to_datetime(df["timestamp"])
    return pd.DataFrame(
        {
            "zone_id": df["zone_id"].astype(int),
            "year": ts.dt.year,
            "month": ts.dt.month,
            "quarter": ts.dt.quarter,
        },
        index=df.index,
    )

model_b2 = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
)
model_b2.fit(target_table_features(train_df), y_train)
report(
    "2. XGBoost (Target Table Only)",
    model_b2.predict_proba(target_table_features(val_df))[:, 1],
    model_b2.predict_proba(target_table_features(test_df))[:, 1],
)

2. XGBoost (Target Table Only)           | Val AUC: 0.7656 | Test AUC: 0.7636


## 5. Baseline 3 — Deep Feature Synthesis (Automatic + Manual Multi-Table Features)

In [12]:
global_rate = float(y_train.mean())
global_maj = int(global_rate >= 0.5)

zone_rates_tr = train_df.groupby("zone_id")["has_damage"].mean()
zone_maj_tr = (zone_rates_tr >= 0.5).astype(int)

train_val_df = pd.concat([train_df, val_df], ignore_index=True)
zone_rates_trval = train_val_df.groupby("zone_id")["has_damage"].mean()
zone_maj_trval = (zone_rates_trval >= 0.5).astype(int)


In [10]:
from test_xgboost import auto_dfs_features, manual_dfs_features, prepare_xgb_matrix

all_rows = pd.concat(
    [
        train_df[["zone_id", "timestamp"]],
        val_df[["zone_id", "timestamp"]],
        test_df[["zone_id", "timestamp"]],
    ],
    ignore_index=True,
)
n_tr, n_va = len(train_df), len(val_df)

auto_all = auto_dfs_features(db, all_rows)
auto_cols = [c for c in auto_all.columns if c not in ("zone_id", "timestamp")]
auto_cats = [c for c in auto_cols if not pd.api.types.is_numeric_dtype(auto_all[c])]

model_b3_auto = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, enable_categorical=True,
    eval_metric="auc", random_state=42, n_jobs=-1,
)
model_b3_auto.fit(prepare_xgb_matrix(auto_all.iloc[:n_tr], auto_cols, auto_cats), y_train)
report(
    "3a. XGBoost (Automatic DFS)",
    model_b3_auto.predict_proba(prepare_xgb_matrix(auto_all.iloc[n_tr:n_tr+n_va], auto_cols, auto_cats))[:, 1],
    model_b3_auto.predict_proba(prepare_xgb_matrix(auto_all.iloc[n_tr+n_va:], auto_cols, auto_cats))[:, 1],
)

man_all = manual_dfs_features(db, all_rows)
man_cols = [c for c in man_all.columns if c not in ("zone_id", "timestamp")]
comb_all = pd.concat([auto_all, man_all[man_cols]], axis=1)
comb_cols = auto_cols + man_cols

model_b3_comb = xgb.XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, enable_categorical=True,
    eval_metric="auc", random_state=42, n_jobs=-1,
)
model_b3_comb.fit(prepare_xgb_matrix(comb_all.iloc[:n_tr], comb_cols, auto_cats), y_train)
report(
    "3b. XGBoost (Manual + Auto DFS)",
    model_b3_comb.predict_proba(prepare_xgb_matrix(comb_all.iloc[n_tr:n_tr+n_va], comb_cols, auto_cats))[:, 1],
    model_b3_comb.predict_proba(prepare_xgb_matrix(comb_all.iloc[n_tr+n_va:], comb_cols, auto_cats))[:, 1],
)

3a. XGBoost (Automatic DFS)              | Val AUC: 0.8230 | Test AUC: 0.8002
3b. XGBoost (Manual + Auto DFS)          | Val AUC: 0.8789 | Test AUC: 0.8564


## 6. Summary Comparison

In [11]:
pd.DataFrame(results_table).T.sort_values("Val ROC-AUC", ascending=False)

,Val ROC-AUC,Test ROC-AUC
3b. XGBoost (Manual + Auto DFS),0.878851,0.856403
3a. XGBoost (Automatic DFS),0.823024,0.800199
1c. Per-Zone Historical Rate,0.806759,0.803726
2. XGBoost (Target Table Only),0.765625,0.763591
1b. Per-Zone Historical Majority,0.637507,0.631780
1a. Global Historical Majority,0.500000,0.500000
